# Dashboard architecture tour

Runs the dashboard's data layer step by step on the real `data/clean/` files, with **no Streamlit involved**. It follows the same order `app.py` uses on every rerun (see `docs/LIFE_OF_A_RERUN.md`):

1. load  →  2. filter  →  3. aggregate  →  4. fill gaps with IDW  →  5. analytics  →  6. AQI

Run the cells top to bottom. Each section ends with a short "what to notice".

Needs `data/clean/` to exist (run `python run_pipeline.py --skip-api` first).

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Find the repo root (the folder containing aqi.py) so imports work from notebooks/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "aqi.py").exists())
sys.path.insert(0, str(ROOT))
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
ROOT

## 1. Load  (`streamlit_app/loading.py`)

`load_pipeline_data` reads the four pipeline outputs and checks their columns. In the app this call is wrapped in `st.cache_data`, so it runs once per server process.

In [ ]:
from streamlit_app.loading import load_pipeline_data

data = load_pipeline_data(ROOT)
print("merged     :", data.merged.shape, "  one row = one sensor on one day")
print("complaints :", data.complaints.shape, "  one row = one 311 complaint")
print("summary    :", data.summary.shape, "  one row = one neighborhood (all-time)")
print("geojson    :", len(data.neighborhoods_geojson["features"]), "neighborhood polygons")
data.merged.head()

**What to notice:** `merged` is the workhorse table. Everything the dashboard shows is an aggregate of it, plus `complaints` for the map's complaint dots.

## 2. Filter  (`streamlit_app/metrics.py`)

The sidebar's date range and neighborhood picker become one call. Here we take the last 30 days in the data. Set `hoods` to a list such as `["Hyde Park"]` to see the neighborhood filter.

In [ ]:
from streamlit_app.metrics import filter_by_date_range

end = data.merged["date"].max()
start = end - pd.Timedelta(days=29)
hoods = []  # empty = all neighborhoods, same as the sidebar default

filtered = filter_by_date_range(data.merged, start, end, hoods)
print(f"{start:%Y-%m-%d} to {end:%Y-%m-%d}: {len(filtered):,} of {len(data.merged):,} sensor-days")

## 3. Aggregate  (`streamlit_app/metrics.py`)

Three views of the same filtered rows: per day (trend charts, KPIs), per sensor (map markers, heatmap) and per neighborhood (choropleth).

In [ ]:
from streamlit_app.metrics import build_city_daily_metrics, build_neighborhood_metrics, build_sensor_snapshot

city_daily = build_city_daily_metrics(filtered)

# app.py then re-indexes to every calendar day so days without readings show as gaps, not skipped days.
full_days = pd.date_range(start, end, freq="D")
city_daily = city_daily.set_index("date").reindex(full_days).rename_axis("date").reset_index()
city_daily["complaint_count"] = city_daily["complaint_count"].fillna(0)

sensors = build_sensor_snapshot(filtered)
base_metrics = build_neighborhood_metrics(data.summary, filtered)

print("city_daily:", city_daily.shape, "| sensors:", sensors.shape, "| neighborhoods:", base_metrics.shape)
city_daily.head()

In [ ]:
# Neighborhoods are kept even when they have no readings in the window; their *_period values are NaN.
base_metrics[["neighborhood", "sensor_count", "pm25_mean_period", "no2_mean_period", "complaints_period"]].sort_values(
    "pm25_mean_period", ascending=False
).head(8)

**What to notice:** `build_neighborhood_metrics` starts from the *summary* table, so all neighborhoods survive the filter. That is why some rows above would have NaN averages.

## 4. Fill gaps with IDW  (`streamlit_app/estimation.py`)

Neighborhoods with no direct sensor values get an inverse-distance-weighted estimate from the *k* nearest sensors. The three sliders in the sidebar's "Neighborhood estimation" section are these three arguments.

In [ ]:
from streamlit_app.estimation import enrich_neighborhood_metrics_with_estimates

map_metrics = enrich_neighborhood_metrics_with_estimates(
    base_metrics, sensors, data.neighborhoods_geojson, k=3, idw_power=2.0, max_distance_km=15.0
)
print(map_metrics["coverage_source"].value_counts().to_string())

estimated = map_metrics[map_metrics["coverage_source"] == "estimated_idw"]
estimated[["neighborhood", "estimated_sensor_count", "estimated_nearest_km", "pm25_mean_estimated", "pm25_map_value"]].head(8)

**What to notice:** `*_map_value` is what the map paints. It equals the direct value when there is one and the IDW estimate otherwise; `coverage_source` records which. Try `max_distance_km=2` above and watch neighborhoods move to `unavailable`.

## 5. Analytics  (`streamlit_app/analytics.py`)

Both functions work on the city-daily table. A positive lag means complaints *trail* PM2.5.

In [ ]:
from streamlit_app.analytics import compute_lag_correlations, compute_spike_concordance

lag = compute_lag_correlations(city_daily, max_lag=7)
best = lag.dropna().loc[lambda d: d["correlation"].abs().idxmax()]
print(f"Strongest absolute lag: {best['lag_days']:.0f} days (correlation {best['correlation']:.3f})")

spikes, baseline = compute_spike_concordance(city_daily, spike_percentile=0.80, window_days=2)
print(f"Non-spike baseline: {baseline:.2f} complaints/day")
spikes

## 6. AQI  (`aqi.py`)

The only module shared by the pipeline and the dashboard. The banner at the top of the app is `pm25_to_aqi` of the window's average PM2.5, plus `aqi_category` and `aqi_health_message`.

In [ ]:
from aqi import aqi_category, aqi_health_message, pm25_to_aqi

avg_pm25 = filtered["pm25_mean"].mean()
aqi = pm25_to_aqi(avg_pm25)
label, color = aqi_category(aqi)
print(f"Average PM2.5 {avg_pm25:.2f} ug/m3  ->  AQI {aqi:.0f}  ({label}, {color})")
print(aqi_health_message(aqi))

## Where next

- `docs/ARCHITECTURE.md`: module map and glossary.
- `docs/LIFE_OF_A_RERUN.md`: how these same steps run inside Streamlit.
- `docs/HOW_TO.md`: recipes for extending the dashboard.